In [11]:
import pandas as pd
import numpy as np

In [12]:
df = pd.read_csv("play_tennis.csv")

In [13]:
print(df.head())

  day   outlook  temp humidity    wind play
0  D1     Sunny   Hot     High    Weak   No
1  D2     Sunny   Hot     High  Strong   No
2  D3  Overcast   Hot     High    Weak  Yes
3  D4      Rain  Mild     High    Weak  Yes
4  D5      Rain  Cool   Normal    Weak  Yes


In [14]:
import math

In [30]:
count_yes = (df['play'] == 'Yes').sum()
print(count_yes)


9


In [31]:
count_no = (df['play'] == "No").sum()

In [32]:
total_count = count_yes + count_no
p_yes = count_yes/total_count
p_no = count_no/total_count
full_entropy = -p_yes*math.log2(p_yes) - p_no*math.log2(p_no)

In [35]:
attribute_gains = {}

In [39]:
for y in df.columns[1:-1]:
  unique = df[y].unique()
  entropy = []
  weights = []
  for i in unique:
    df_attribute = df[df[y] == i]
    count_yes = (df_attribute['play'] == 'Yes').sum()
    count_no = (df_attribute['play'] == 'No').sum()
    tot = count_yes + count_no
    p_yes = count_yes/tot
    p_no = count_no/tot
    if(p_yes == 0 or p_no == 0):
      attribute_entropy = 1
    else :
      attribute_entropy = -p_yes*math.log2(p_yes) - p_no*math.log2(p_no)
    entropy.append(attribute_entropy)
    weight = (df[y] == i).sum()
    weights.append(weight/total_count)
    gain = full_entropy - np.sum(np.array(entropy)*np.array(weights))
    attribute_gains[y] = gain

In [40]:
print(attribute_gains)

{'outlook': np.float64(-0.03896446593984637), 'temp': np.float64(0.02922256565895487), 'humidity': np.float64(0.15183550136234159), 'wind': np.float64(0.04812703040826949)}


In [41]:
max_attribute = "placeholder"
max_val = 0
for i in attribute_gains:
  if(attribute_gains[i] > max_val):
    max_val = attribute_gains[i]
    max_attribute = i

root = max_attribute

In [45]:
def calculate_entropy(df):
    target_column = 'play'
    count_yes = (df[target_column] == 'Yes').sum()
    count_no = (df[target_column] == 'No').sum()
    total_count = count_yes + count_no

    if total_count == 0:
        return 0.0

    p_yes = count_yes / total_count
    p_no = count_no / total_count

    entropy = 0.0
    if p_yes > 0:
        entropy -= p_yes * math.log2(p_yes)
    if p_no > 0:
        entropy -= p_no * math.log2(p_no)
    return entropy

def build_tree(df):
    target_column = 'play'

    # Base Case 1: If all instances in the current DataFrame have the same 'play' value
    if len(df[target_column].unique()) == 1:
        return df[target_column].iloc[0]

    # Base Case 2: If there are no more attributes to split on (excluding 'day' and 'play')
    # We drop 'day' because it's usually an identifier and 'play' is the target.
    features = [col for col in df.columns if col not in ['day', target_column]]
    if not features:
        # Return the majority class if no features left to split
        return df[target_column].mode()[0]

    full_entropy = calculate_entropy(df)
    attribute_gains = {}

    for attribute in features:
        unique_values = df[attribute].unique()
        weighted_average_entropy = 0.0
        for value in unique_values:
            subset = df[df[attribute] == value]
            subset_entropy = calculate_entropy(subset)
            weight = len(subset) / len(df)
            weighted_average_entropy += weight * subset_entropy

        gain = full_entropy - weighted_average_entropy
        attribute_gains[attribute] = gain

    # Find the attribute with the maximum information gain
    best_attribute = max(attribute_gains, key=attribute_gains.get)

    # Create a node for the best attribute
    tree = {best_attribute: {}}

    # Recursively build the tree for each unique value of the best attribute
    for value in df[best_attribute].unique():
        subset = df[df[best_attribute] == value].copy()
        # Drop the chosen attribute for the next level of recursion
        subset_for_recursion = subset.drop(columns=[best_attribute])
        tree[best_attribute][value] = build_tree(subset_for_recursion)

    return tree

# Build the decision tree starting with the original DataFrame
decision_tree = build_tree(df)
print(decision_tree)

{'outlook': {'Sunny': {'humidity': {'High': 'No', 'Normal': 'Yes'}}, 'Overcast': 'Yes', 'Rain': {'wind': {'Weak': 'Yes', 'Strong': 'No'}}}}


In [46]:
def print_tree(tree, indent=" "):
    for key, value in tree.items():
        if isinstance(value, dict):
            print(indent + str(key) + ":")
            print_tree(value, indent + "  ")
        else:
            print(indent + str(key) + ": " + str(value))

print("Decision Tree:")
print_tree(decision_tree)

Decision Tree:
 outlook:
   Sunny:
     humidity:
       High: No
       Normal: Yes
   Overcast: Yes
   Rain:
     wind:
       Weak: Yes
       Strong: No


In [42]:
print(root)

humidity


In [43]:
branches = df[root].unique()

In [44]:
print(branches)

['High' 'Normal']


In [ ]:
for i in branches:
  df_branch = df[df[root] == i]
  df_branch.drop(root, axis = 1, inplace = True)

